In [ ]:
import pickle 
import matplotlib.pyplot as plt
import pandas as pd 
import numpy as np 

In [ ]:
!pip install statannotations
from statannotations.Annotator import Annotator

In [ ]:
key = "" # sample identifier

In [ ]:
cell_clustering_path = r"" # path to cell phenotyping results

In [ ]:
fibrosis_path = r"" # path to fibrosis clustering results

In [ ]:
with open(fibrosis_path + '/wsi_labeled.pickle', 'rb') as f:
    wsi_labeled = pickle.load(f)

In [ ]:
with open(fibrosis_path + '/cluster_color_dict.pickle', 'rb') as f:
    cluster_color_dict = pickle.load(f)

In [ ]:
with open(cell_clustering_path + '/segmentation_mask_merged_within_contour_for_pixel_clustering.pickle', 'rb') as f: 
    cell_segmentation_mask = pickle.load(f)

In [ ]:
fibrosis_som_clusters = pd.read_csv(fibrosis_path + '/fibrosis_density_som_clusters.csv').drop(columns=['Unnamed: 0'])

In [ ]:
import seaborn as sns 

In [ ]:
fibrosis_som_clusters = fibrosis_som_clusters.rename(columns={'pixel_som_cluster': 'Fibrosis Clusters'})

In [ ]:
sns.set(font_scale=1) 
sns.clustermap(fibrosis_som_clusters.groupby('Fibrosis Clusters').median().drop(columns=['X', 'Y']).transpose(), figsize=(5, 2), yticklabels=True, row_cluster=False, cbar_kws={'location': 'right'})
plt.savefig(fibrosis_path + '/clusters_heatmap.png')

In [ ]:
ytick_labels = ['i-IFTA', 'Established Fibrosis', 'Diffuse/Limited Fibrosis']

In [ ]:
import matplotlib as mpl

In [ ]:
mpl.rcParams["font.size"] = 80
tile_size = 512
height, width = cell_segmentation_mask.shape

fibrosis_som_clusters['tile_x'] = fibrosis_som_clusters['X'] // tile_size
fibrosis_som_clusters['tile_y'] = fibrosis_som_clusters['Y'] // tile_size

# Count pixels per cluster per tile
tile_cluster_counts = (
    fibrosis_som_clusters
    .groupby(['tile_y', 'tile_x', 'Fibrosis Clusters'])
    .size()
    .reset_index(name='pixel_count')
)

pivot = tile_cluster_counts.pivot_table(
    index=['tile_y', 'tile_x'],
    columns='Fibrosis Clusters',
    values='pixel_count',
    fill_value=0
)

pivot_fraction = pivot.div(pivot.sum(axis=1), axis=0)

# Mean ± stderr across tiles
mean_per_cluster = pivot_fraction.mean(axis=0)
std_per_cluster = pivot_fraction.std(axis=0)
num_tiles = len(pivot_fraction)
stderr_per_cluster = std_per_cluster / np.sqrt(num_tiles)

colors = [cluster_color_dict[c] for c in mean_per_cluster.index]

indices = [1, 2, 3]

plt.figure(figsize=(6, 4))
ax = plt.gca()

ax.barh(
    indices,
    mean_per_cluster.values,
    xerr=stderr_per_cluster.values,
    capsize=5,
    color=colors, height=0.5
)

ax.set_facecolor('white')
plt.gcf().patch.set_facecolor('white')

ax.grid(False)


ax.set_yticks(indices)
ax.set_yticklabels(ytick_labels)

ax.set_xlabel('Fraction Pixels Assigned')
ax.set_ylabel('Cluster')

plt.show()
